In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import cobra

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

# First, import the base kinGEMs package
import kinGEMs

# Import necessary functions from kinGEMs package
from kinGEMs.dataset import load_model, convert_to_irreversible, prepare_model_data, retrieve_sequences, map_metabolites, merge_substrate_sequences
from kinGEMs.modeling.optimize import run_optimization
from kinGEMs.plots import plot_flux_distribution  # Assuming this function exists

# Define paths
data_dir = "../data"
raw_data_dir = os.path.join(data_dir, "raw")
model_path = os.path.join(raw_data_dir, "e_coli_core.xml")
interim_data_dir = os.path.join(data_dir, "interim")
processed_data_dir = os.path.join(data_dir, "processed")

# Step 1: Load the model
print("Loading model...")
model = load_model(model_path)
print(f"Model loaded with {len(model.reactions)} reactions and {len(model.metabolites)} metabolites")

# Step 2: Convert to irreversible model
print("Converting to irreversible model...")
irrev_model = convert_to_irreversible(model)
print(f"Irreversible model created with {len(irrev_model.reactions)} reactions")

# Step 3: Prepare model data (metabolite and sequence mapping)
print("Preparing model data...")
# Define output paths
substrates_output = os.path.join(interim_data_dir, "e_coli_substrates.csv")
sequences_output = os.path.join(interim_data_dir, "e_coli_sequences.csv")

# Prepare model data
irrev_model, substrate_df, sequences_df = prepare_model_data(
    model_path=model_path,
    substrates_output=substrates_output,
    sequences_output=sequences_output,
    organism='E coli'
)

# Step 4: Merge substrate and sequence data
print("Merging substrate and sequence data...")
merged_data_output = os.path.join(processed_data_dir, "e_coli_merged_data.csv")
merged_data = merge_substrate_sequences(
    substrate_df=substrate_df,
    sequences_df=sequences_df,
    model=irrev_model,
    output_path=merged_data_output
)

# Step 5: Run optimization
print("Running optimization...")
# Set parameters for optimization
biomass_reaction = 'BIOMASS_Ec_iML1515_core_75p37M'  # Update with your biomass reaction ID
enzyme_upper_bound = 0.3  # gP/gDCW, enzyme fraction upper bound

# Run optimization
solution, flux_distribution = run_optimization(
    model=irrev_model,
    kcat_dict=merged_data,
    biomass_reaction=biomass_reaction,
    enzyme_upper_bound=enzyme_upper_bound,
    enzyme_ratio=True  # Using enzyme fraction constraint
)

print(f"Optimization complete. Biomass flux: {solution.objective_value}")

# Step 6: Plot results
print("Plotting flux distribution...")
# Plot flux distribution
fig = plot_flux_distribution(flux_distribution)
plt.show()

# Save results
flux_distribution.to_csv(os.path.join(processed_data_dir, "e_coli_flux_distribution.csv"), index=False)

print("Pipeline completed successfully!")

2025-03-18 17:14:25.752 | INFO     | kinGEMs.config:<module>:12 - PROJ_ROOT path is: C:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2


Loading model...
Set parameter Username
Academic license - for non-commercial use only - expires 2026-02-25
Model loaded with 95 reactions and 72 metabolites
Converting to irreversible model...
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp2j9frmnv.lp
Reading time = 0.01 seconds
: 72 rows, 190 columns, 720 nonzeros
Number of reactions that are non-exchange:  75
Number of reactions that are exchange:  20
Number of reactions being added from non-exchange: 39
Number of reactions being added from exchange: 59
Irreversible model created with 154 reactions
Preparing model data...
Loaded model with 95 reactions and 72 metabolites
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmpswqqg3dx.lp
Reading time = 0.00 seconds
: 72 rows, 190 columns, 720 nonzeros
Number of reactions that are non-exchange:  75
Number of reactions that are exchange:  20
Number of reactions being added from non-exchange: 39
Number of reactions being added from exchange: 59
Converted 

c:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2\kinGEMs\dataset.py:197: DtypeWarning: Columns (4,10) have mixed types. Specify dtype option on import or set low_memory=False.
  SEED_comps = pd.read_csv(SEED_COMPOUNDS, sep='\t')


There are 71 substrates in the GEM.
-----------------------------
Mapping substrate: atp_c
BiGG Name: ATP C10H12N5O13P3
SMILES found in MetaNetX: Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O
-----------------------------
Mapping substrate: f6p_c
BiGG Name: D-Fructose 6-phosphate
SMILES found in SEED: O=P([O-])([O-])OC[C@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O
-----------------------------
Mapping substrate: coa_c
BiGG Name: Coenzyme A
-----------------------------
Mapping substrate: pyr_c
BiGG Name: Pyruvate
SMILES found in MetaNetX: CC(=O)C(=O)[O-]
-----------------------------
Mapping substrate: g6p_c
BiGG Name: D-Glucose 6-phosphate
SMILES found in SEED: O=P([O-])([O-])OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O
-----------------------------
Mapping substrate: 3pg_c
BiGG Name: 3-Phospho-D-glycerate
SMILES found in SEED: O=C([O-])[C@H](O)COP(=O)([O-])[O-]
-----------------------------
Mapping substrate: 6pgl_c
BiGG Name: 6-phospho-D-glucono-1,5-lactone
S

-----------------------------
Mapping substrate: f6p_c


...

-----------------------------
Mapping substrate: actp_c


ERROR:root:Unexpected error retrieving SMILES for actp_c: [WinError 10054] An existing connection was forcibly closed by the remote host
Traceback (most recent call last):
  File "c:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2\kinGEMs\dataset.py", line 472, in get_SMILES_from_cactus
    cmpd_smiles = urlopen(url, timeout=10).read().decode('utf8').strip()
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 214, in urlopen
    return opener.open(url, data, timeout)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 517, in open
    response = self._open(req, data)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 534, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 1375, i

-----------------------------
Mapping substrate: succoa_c


ERROR:root:Unexpected error retrieving SMILES for succoa_c: [WinError 10054] An existing connection was forcibly closed by the remote host
Traceback (most recent call last):
  File "c:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2\kinGEMs\dataset.py", line 472, in get_SMILES_from_cactus
    cmpd_smiles = urlopen(url, timeout=10).read().decode('utf8').strip()
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 214, in urlopen
    return opener.open(url, data, timeout)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 517, in open
    response = self._open(req, data)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 534, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 494, in _call_chain
    result = func(*args)
  File "c:\Users\Rana\Anaconda3\envs\met_modeling\lib\urllib\request.py", line 1375,

Mapped metabolites to SMILES (263 found)
